In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import time
import torch
import random
import numpy as np
from glob import glob
from pathlib import Path
from src.utilities.report import Report
from src.service.fragment.net import Net
from src.utilities.score_mapper import ScoreMapper
from src.utilities.load_dataset import load_dataset
from skl2onnx.helpers.onnx_helper import load_onnx_model
from src.service.stitching.generate_networks import generate_networks
from src.utilities.dataloader_generator import generate_dataloader

In [3]:
netsFiles = sorted(glob('../_results_with_finetune/fragments/net*'))
nets = []
for index, netsFile in enumerate(netsFiles):
    fragmentFiles = sorted(glob(str(Path(netsFile)/'fragment*.onnx')))
    onnxFragments = []
    for fragmentFile in fragmentFiles:
        onnxFragment = load_onnx_model(fragmentFile)
        onnxFragments.append(onnxFragment)
    net1 = Net(onnxFragments, index)
    nets.append(net1)

In [4]:
random.seed(51)
np.random.seed(24)
torch.manual_seed(77)

K = 5
STITCH_BATCH_SIZE = 32 # todo study the effect
MAX_DEPTH = 16
THRESOULD = 0
TOTAL_THRESOULD = 0.5

RESULT_NAME = f"{int(time.time())}_result_BS_{STITCH_BATCH_SIZE}_MD_{MAX_DEPTH}_T_{THRESOULD}_TT_{TOTAL_THRESOULD}_K_{K}"

EVAL_BATCH_SIZE = 64

train_datalader = generate_dataloader(load_dataset(), batch_size=STITCH_BATCH_SIZE)
test_datalader = generate_dataloader(load_dataset("test"), batch_size=EVAL_BATCH_SIZE)
data_score, _ = next(iter(train_datalader))
data_score = data_score.numpy()
print(data_score.shape)

(32, 3, 224, 224)


In [5]:
# range(1)
k = 0
if os.path.exists(f'../_results_with_finetune/{RESULT_NAME}.txt'):
    with open(f'../_results_with_finetune/{RESULT_NAME}.txt', 'r') as f:
        k = len(f.read().split('\n'))
        print(k)

In [6]:
scoreMapper = ScoreMapper(nets, data_score)
with Report(EVAL_BATCH_SIZE, f'./_results_with_finetune/{RESULT_NAME}.txt', 'a') as report:
    generator = generate_networks(nets, scoreMapper, data_score, 
                          threshold=THRESOULD, totalThreshold=TOTAL_THRESOULD, 
                          maxDepth=MAX_DEPTH, sample=False, K=K)
    for i,(s,net) in enumerate(generator):
        try:
            netname = f"../_results_with_finetune/{RESULT_NAME}/net{k:03}"
            report.evaluate(nets, net, netname, s, test_datalader)
            net.save(netname)
            k += 1
        except Exception as e:
            print('ERROR', e)
            pass

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:121 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:114 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] CUDA failure 900: operation not permitted when stream is capturing ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/cuda_execution_provider.cc ; line=248 ; expr=cudaDeviceSynchronize(); 

 when using ['CUDAExecutionProvider', 'CPUExecutionPr

2024-04-28 11:47:16.889417093 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running FusedConv node. Name:'/features/features.28/Conv' Status Message: CUDA error cudaErrorStreamCaptureUnsupported:operation not permitted when stream is capturing
Exception in thread Thread-9 (net_scores):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/shafigh/Desktop/projects/stitchnet/.stitchnet/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/shafigh/Desktop/projects/stitchnet/src/utilities/score_mapper.py", line 21, in net_scores
    nextscores = net2.get_scores(x1, data, self.scoring_method)
  File "/home/shafigh/Desktop/projects/stitchnet/src/service/fragment/net.py", line 209, in get_

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:121 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] /onnxruntime_src/onnxruntime/core/providers/cuda/cuda_call.cc:114 std::conditional_t<THRW, void, onnxruntime::common::Status> onnxruntime::CudaCall(ERRTYPE, const char*, const char*, ERRTYPE, const char*, const char*, int) [with ERRTYPE = cudaError; bool THRW = true; std::conditional_t<THRW, void, onnxruntime::common::Status> = void] CUDA failure 900: operation not permitted when stream is capturing ; GPU=0 ; hostname=LIA-WS-045 ; file=/onnxruntime_src/onnxruntime/core/providers/cuda/cuda_execution_provider.cc ; line=248 ; expr=cudaDeviceSynchronize(); 

 when using ['CUDAExecutionProvider', 'CPUExecutionPr

Exception in thread Thread-7 (net_scores):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/shafigh/Desktop/projects/stitchnet/.stitchnet/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/shafigh/Desktop/projects/stitchnet/src/utilities/score_mapper.py", line 21, in net_scores
    nextscores = net2.get_scores(x1, data, self.scoring_method)
  File "/home/shafigh/Desktop/projects/stitchnet/src/service/fragment/net.py", line 209, in get_scores
    x2 = self.get_output(fragment, data)
  File "/home/shafigh/Desktop/projects/stitchnet/src/service/fragment/net.py", line 237, in get_output
2024-04-28 11:47:17.680168154 [E:onnxruntime:Default, cuda_call.cc:116 CudaCall] CUBLAS failure 13: CUBLAS_STATUS_EXECUTION_FAILED ; GPU=0 ; h

potential next fragments before thresholding of 0: 5 ['1.0', '0.71', '0.65', '0.64', '0.61']
potential next fragments after thresholding of 0: 5 ['1.0', '0.71', '0.65', '0.64', '0.61']
totalscore before thresholding of 0.5: 1.0


...

2024-04-28 13:07:04.799420286 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running FusedConv node. Name:'/features/features.0/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 411041792



[WARNING] [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running FusedConv node. Name:'/features/features.0/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 411041792

totalscore before thresholding of 0.5: 0.5731939077377319


2024-04-28 13:07:07.673489000 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running FusedConv node. Name:'/features/features.0/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 411041792



[WARNING] [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running FusedConv node. Name:'/features/features.0/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 411041792

totalscore before thresholding of 0.5: 0.5164088010787964
[WARNING] [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running FusedConv node. Name:'/features/features.0/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 411041792

totalscore before thresholding of 0.5: 0.44308727979660034
totalscore before thresholding of 0.5: 0.432472258806

2024-04-28 13:07:10.655455941 [E:onnxruntime:, sequential_executor.cc:514 ExecuteKernel] Non-zero status code returned while running FusedConv node. Name:'/features/features.0/Conv' Status Message: /onnxruntime_src/onnxruntime/core/framework/bfc_arena.cc:376 void* onnxruntime::BFCArena::AllocateRawInternal(size_t, bool, onnxruntime::Stream*, bool, onnxruntime::WaitNotificationFn) Failed to allocate memory for requested buffer of size 411041792

